# 워크북 0 — 공개 감성 리뷰 데이터 수집
**실습 목표:** 허용된 공개 원자료 확인 → 다운로드 → 데이터 품질 검증 → 100건 균형 샘플 → 수집이력 보관.
**주의:** NSMC는 영화 리뷰로 쇼핑몰 리뷰가 아닙니다. 학습된 감성 기준 차이를 토론하세요. 웹사이트 무단 크롤링은 하지 않습니다.


## 1. 공개 데이터 다운로드


In [ ]:
import pandas as pd
from pathlib import Path
url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
df = pd.read_csv(url, sep="\t", dtype={"id":"string"})
print("원본 행 수:", len(df))
display(df.head())
print(df.isna().sum())
print(df["label"].value_counts(dropna=False))


## 2. 정제 및 100개 샘플 생성


In [ ]:
clean = df.dropna(subset=["document","label"]).drop_duplicates(subset=["id"]).copy()
clean["document"] = clean["document"].astype(str).str.strip()
clean = clean[clean["document"].str.len() > 0]
clean["sentiment"] = clean["label"].map({0:"부정", 1:"긍정"})
sample = pd.concat([
    clean[clean["label"]==0].sample(n=50, random_state=42),
    clean[clean["label"]==1].sample(n=50, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)
sample = sample.rename(columns={"id":"review_id","document":"review_text","label":"original_label"})
sample["source"] = "NSMC_movie"
sample.to_csv("week06_nsmc_collected_100.csv", index=False, encoding="utf-8-sig")
display(sample.head());print("샘플:",len(sample))


## 3. 출처 기록


In [ ]:
# 수집 이력 저장: 출처와 방법은 데이터셋에 반드시 기록합니다.
from datetime import datetime, timezone
import json
record={"dataset":"NSMC 영화 리뷰","source_url":url,"collected_at_utc":datetime.now(timezone.utc).isoformat(),
        "original_count":len(df),"sample_count":len(sample),"method":"CSV/TSV 공개 파일 직접 다운로드",
        "note":"영화 리뷰(긍/부정)이며 쇼핑 리뷰(긍/부정/중립)와 도메인/레이블 차이 있음"}
Path("week06_collection_record.json").write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding="utf-8")
print(record)


## 결과물
`week06_nsmc_collected_100.csv`, `week06_collection_record.json`을 다운로드해 제출하고, 쇼핑몰 합성 리뷰와 비교할 때 중립 라벨 유무를 설명하세요.
